# Alytes-ReID — Toad Identification Tool

**Who uses this**: field biologist (daily use).

**What it does**: upload a photo → get the matching individual from the database.

**Prerequisites**: run `01_setup_and_training.ipynb` once to train the models and save them to Google Drive.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danort92/Alytes-ReID/blob/claude/alytes-reid-system-qgved/notebooks/02_toad_reid.ipynb)

---
## 1. Setup (run once per session)

Takes ~2 minutes. After this, the identification cells below can be re-run as many times as needed without restarting.

In [ ]:
# Install and load the project
!git clone -b claude/alytes-reid-system-qgved https://github.com/danort92/Alytes-ReID.git 2>/dev/null || true
%cd Alytes-ReID
!pip install -r requirements.txt -q

# Mount Google Drive (where trained models are stored)
from google.colab import drive
drive.mount('/content/drive')

print('Setup complete!')

In [ ]:
import shutil
from pathlib import Path

DRIVE_MODEL_DIR = '/content/drive/MyDrive/Alytes-ReID/models'

# Copy models from Drive to local (fast access)
local_models = Path('data/models')
local_models.mkdir(parents=True, exist_ok=True)

detection_model = local_models / 'detection_best.pt'
reid_model_path = local_models / 'reid_model.pt'
reid_db_path    = local_models / 'reid_db'

shutil.copy2(f'{DRIVE_MODEL_DIR}/detection_best.pt', str(detection_model))
print(f'Detection model loaded ✓')

if Path(f'{DRIVE_MODEL_DIR}/reid_model.pt').exists():
    shutil.copy2(f'{DRIVE_MODEL_DIR}/reid_model.pt', str(reid_model_path))
    print(f'Re-ID model loaded ✓')
else:
    reid_model_path = None
    print('Re-ID model not found (train it in notebook 01)')

if Path(f'{DRIVE_MODEL_DIR}/reid_db').exists():
    shutil.copytree(f'{DRIVE_MODEL_DIR}/reid_db', str(reid_db_path), dirs_exist_ok=True)
    print(f'Embedding database loaded ✓')
else:
    print('No database yet — you can create it by registering new individuals below')

In [ ]:
import torch
import cv2
import matplotlib.pyplot as plt
from src.detection.predict import load_detector
from src.segmentation.segment import ToadSegmenter
from src.reid.model import build_model
from src.reid.database import EmbeddingDatabase
from src.utils.io import load_yaml

det_config  = load_yaml(Path('config/detection.yaml'))
pre_config  = load_yaml(Path('config/preprocessing.yaml'))
reid_config = load_yaml(Path('config/reid.yaml'))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

# Detection
detector  = load_detector(detection_model)
segmenter = ToadSegmenter(device=device)

# Re-ID
reid_model = build_model(reid_config).to(device)
if reid_model_path:
    reid_model.load_state_dict(torch.load(str(reid_model_path), map_location=device))
reid_model.eval()

# Database
db = EmbeddingDatabase(
    embedding_dim=reid_config['model']['embedding_dim'],
    index_type=reid_config['database']['index_type'],
)
if reid_db_path.exists() and (reid_db_path / 'index.faiss').exists():
    db.load(reid_db_path)
    print(f'Database: {db.size} embeddings — {len(db.get_individuals())} individuals')
else:
    print('Empty database — register individuals to start.')

print('Models ready ✓')

---
## 2. Identify a Toad

Upload a dorsal photo and the system will return the top matching individuals.

In [ ]:
from google.colab import files as colab_files
from src.detection.predict import detect_toads, get_best_detection
from src.preprocessing.pipeline import preprocess_toad
from src.reid.match import match_toad
from src.utils.visualization import draw_detections, draw_mask_overlay

print('Upload a toad photo:')
uploaded = colab_files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    query_img = cv2.cvtColor(cv2.imread(filename), cv2.COLOR_BGR2RGB)
    query_path = Path(filename)

    # Step 1: Detect
    detections = detect_toads(detector, query_path)
    best_det = get_best_detection(detections)

    if best_det is None:
        print('No toad detected. Try a clearer dorsal photo.')
    else:
        print(f'Toad detected (confidence: {best_det["confidence"]:.2f})')

        # Step 2: Segment
        cropped, mask = segmenter.segment_and_crop(query_img, best_det['bbox'])
        overlay = draw_mask_overlay(
            draw_detections(query_img, detections),
            segmenter.segment_from_bbox(query_img, best_det['bbox'])
        )

        # Step 3: Preprocess
        standardized = preprocess_toad(cropped, mask, pre_config)

        # Step 4: Match
        result = match_toad(
            reid_model, db, standardized,
            top_k=reid_config['inference']['top_k'],
            threshold=reid_config['inference']['similarity_threshold'],
            device=device,
        )

        # Display
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        axes[0].imshow(overlay);      axes[0].set_title('Detection + Mask')
        axes[1].imshow(cropped);      axes[1].set_title('Segmented')
        axes[2].imshow(standardized); axes[2].set_title('Standardized (Re-ID input)')
        for ax in axes: ax.axis('off')
        plt.tight_layout()
        plt.show()

        print('\n' + '='*40)
        if result['is_new']:
            print('  ⚠  POTENTIAL NEW INDIVIDUAL')
            print('     No confident match found in database.')
            print('     Use the Registration cell below to add this toad.')
        else:
            top = result['matches'][0]
            print(f'  ✓  MATCH: {top["individual_id"]}  (score: {top["score"]:.3f})')

        print('\nTop matches:')
        for m in result['matches']:
            bar = '█' * int(m['score'] * 20)
            print(f"  {m['individual_id']:15s} {m['score']:.3f}  {bar}")
        print('='*40)

---
## 3. Register a New Individual

Run this cell if the toad above was flagged as a new individual.

In [ ]:
NEW_ID           = 'TOAD_001'    # ← change this
CAPTURE_DATE     = '2025-04-15'  # ← change this (YYYY-MM-DD)
CAPTURE_LOCATION = 'Site A'      # ← optional

# Make sure you ran the Identify cell above first
try:
    db.add(
        embedding=result['embedding'],
        individual_id=NEW_ID,
        photo_path=str(query_path),
        capture_date=CAPTURE_DATE,
        extra={'location': CAPTURE_LOCATION},
    )
    db.save(reid_db_path)

    # Sync back to Drive immediately
    shutil.copytree(str(reid_db_path), f'{DRIVE_MODEL_DIR}/reid_db', dirs_exist_ok=True)

    print(f'Registered: {NEW_ID}')
    print(f'Database: {db.size} embeddings — {len(db.get_individuals())} individuals')
except NameError:
    print('Run the Identify cell first to get the embedding.')

---
## 4. View Database

In [ ]:
individuals = db.get_individuals()
print(f'Database: {db.size} total embeddings — {len(individuals)} individuals\n')
print(f'{"ID":<20} {"Sightings":>10}  {"Last capture"}')
print('-' * 50)
for ind_id in individuals:
    records = [m for m in db.metadata if m['individual_id'] == ind_id]
    dates = [r.get('capture_date', '') for r in records if r.get('capture_date')]
    last_date = max(dates) if dates else 'unknown'
    print(f'{ind_id:<20} {len(records):>10}  {last_date}')

---
## 5. Batch Processing

Upload multiple photos at once.

In [ ]:
import pandas as pd
from src.utils.io import export_results_csv

print('Upload multiple toad photos:')
batch_uploaded = colab_files.upload()
batch_results = []

for fname, data in batch_uploaded.items():
    img = cv2.cvtColor(cv2.imread(fname), cv2.COLOR_BGR2RGB)
    dets = detect_toads(detector, Path(fname))
    det = get_best_detection(dets)

    if det is None:
        batch_results.append({'photo': fname, 'match': 'NO_DETECTION', 'score': 0.0, 'is_new': True})
        continue

    crop, msk = segmenter.segment_and_crop(img, det['bbox'])
    std = preprocess_toad(crop, msk, pre_config)
    res = match_toad(reid_model, db, std, device=device,
                     top_k=1, threshold=reid_config['inference']['similarity_threshold'])

    best_match = res['matches'][0] if res['matches'] else {}
    batch_results.append({
        'photo': fname,
        'match': best_match.get('individual_id', 'UNKNOWN'),
        'score': round(best_match.get('score', 0.0), 4),
        'is_new': res['is_new'],
    })
    print(f"  {fname}: {'NEW' if res['is_new'] else best_match.get('individual_id','?')} ({best_match.get('score',0):.3f})")

df = pd.DataFrame(batch_results)
print('\n', df.to_string(index=False))

# Export and download
csv_path = Path('results/batch_results.csv')
export_results_csv(batch_results, csv_path)
colab_files.download(str(csv_path))

---
## 6. Export Database for Mark-Recapture (RMark / MARK)

Exports the full sighting history as a CSV compatible with RMark/MARK software.

In [ ]:
from src.utils.io import export_results_csv

export_path = Path('results/sightings_export.csv')
export_results_csv(db.metadata, export_path)

print(f'Exported {len(db.metadata)} sighting records to {export_path}')
colab_files.download(str(export_path))